# Replacement Trap Analysis - Core Calculations

This notebook contains the core analysis: data loading, metrics calculation, and replacement scenarios.

**Dependencies:**
- `lib/replacement_trap_config.py` - Configuration and constants
- `lib/replacement_trap_utils.py` - Calculation functions
- `data/systems-data.json` - System data

**Output:** DataFrame `df` with all calculated metrics ready for visualization and Monte Carlo analysis.

**Methodology Note:** Analysis uses undiscounted cash flows for accessibility and worst-case conservatism.



In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys
import os

# Add lib directory to path for imports
# Bootstrap: find lib directory to import get_notebooks_root
cwd = Path.cwd()
lib_dir = None

# Quick bootstrap to find lib (most common case first)
if (cwd.parent / 'lib' / 'replacement_trap_utils.py').exists() and cwd.name == 'analysis':
    lib_dir = cwd.parent / 'lib'
elif (cwd / 'lib' / 'replacement_trap_utils.py').exists():
    lib_dir = cwd / 'lib'
elif (cwd / 'notebooks' / 'lib' / 'replacement_trap_utils.py').exists():
    lib_dir = cwd / 'notebooks' / 'lib'
else:
    # Walk up to find lib directory
    for parent in [cwd] + list(cwd.parents):
        if (parent / 'lib' / 'replacement_trap_utils.py').exists():
            lib_dir = parent / 'lib'
            break

# Fallback: assume standard structure if we're in analysis/ directory
if lib_dir is None and cwd.name == 'analysis':
    potential_lib = cwd.parent / 'lib'
    if (potential_lib / 'replacement_trap_utils.py').exists():
        lib_dir = potential_lib

# Ensure we found lib_dir before attempting import
if lib_dir is None:
    raise ImportError(
        f"Could not locate lib directory. Current working directory: {cwd}\n"
        f"Expected to find lib/replacement_trap_utils.py relative to notebooks root.\n"
        f"Please ensure you're running this notebook from the notebooks/analysis/ directory."
    )

if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

# Now use the utility function for consistent path detection
import importlib
import replacement_trap_utils as rtu
rtu = importlib.reload(rtu)
get_notebooks_root = rtu.get_notebooks_root

notebooks_root = get_notebooks_root()
lib_dir = notebooks_root / 'lib'

if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

from replacement_trap_config import (
    ELECTRICITY_RATE,
    WATER_RATE,
    DISHWASHER_CYCLES_PER_YEAR,
    DISHWASHER_ANNUAL_LABOR_SAVINGS,
    BASELINES,
    HELOC_RATE,
    HELOC_LOAN_TERM,
    TIME_HORIZON,
    EFFICIENCY_DEGRADATION_RATE,
)

from replacement_trap_utils import (
    calculate_dishwasher_annual_cost,
    calculate_water_heater_annual_cost,
    calculate_ac_annual_cost,
    calculate_payback_period,
    calculate_rp_ratio,
    calculate_lifetime_value_cash,
    calculate_lifetime_value_heloc,
    calculate_scenario_b_rp,
    calculate_lifetime_value_scenario_b_cash,
    calculate_lifetime_value_scenario_b_heloc,
    validate_scenario,
    load_reference_and_compare,
)

# Import validation modules
try:
    from replacement_trap_validation import validate_unit_consistency
    from validate_reference_data import validate_systems_data
    VALIDATION_AVAILABLE = True
except ImportError:
    VALIDATION_AVAILABLE = False
    print("⚠️ Validation modules not available - skipping validation checks")

print("✓ Imports loaded")
print(f"  Electricity rate: ${ELECTRICITY_RATE:.4f}/kWh")
print(f"  Water rate: ${WATER_RATE:.5f}/gal")
print(f"  HELOC rate: {HELOC_RATE*100:.1f}%, term: {HELOC_LOAN_TERM} years")
print(f"  Time horizon: {TIME_HORIZON} years")

# Run unit consistency validation
if VALIDATION_AVAILABLE:
    print("\n🔍 Running unit consistency validation...")
    unit_result = validate_unit_consistency()
    if unit_result.passed:
        print("  ✓ Unit consistency checks passed")
    else:
        print(f"  ❌ Unit consistency checks failed: {len(unit_result.errors)} error(s)")
        for err in unit_result.errors[:3]:  # Show first 3 errors
            print(f"    - {err}")
        if len(unit_result.errors) > 3:
            print(f"    ... and {len(unit_result.errors) - 3} more error(s)")
    if unit_result.warnings:
        print(f"  ⚠️ {len(unit_result.warnings)} warning(s) - review recommended")



ModuleNotFoundError: No module named 'pandas'

## 1. Load System Data

Load system data from JSON and build the main DataFrame with annual savings calculations.



In [ ]:
from pathlib import Path

if 'get_notebooks_root' not in globals():
    def get_notebooks_root():
        """Fallback path resolver so this cell can run without the setup cell."""
        cwd = Path.cwd()

        if (cwd.parent / 'lib' / 'replacement_trap_utils.py').exists() and cwd.name == 'analysis':
            return cwd.parent
        if (cwd / 'lib' / 'replacement_trap_utils.py').exists():
            return cwd
        if (cwd / 'notebooks' / 'lib' / 'replacement_trap_utils.py').exists():
            return cwd / 'notebooks'
        for parent in [cwd] + list(cwd.parents):
            if (parent / 'lib' / 'replacement_trap_utils.py').exists():
                return parent

        return cwd.parent if cwd.name == 'analysis' else cwd

# Load systems data
notebooks_root = get_notebooks_root()
systems_data_path = notebooks_root / 'data' / 'systems-data.json'
with open(systems_data_path, 'r') as f:
    systems_json = json.load(f)

categories = systems_json['appliance_categories']
systems_list = []

# Process Dishwashers (Replacement vs old unit - incremental energy/water savings only)
dishwasher_baseline = BASELINES['Dishwasher']
baseline_dw_kwh = dishwasher_baseline['annual_kWh_point']
baseline_water_gal = dishwasher_baseline['water_gal_per_cycle']
baseline_dw_cost = calculate_dishwasher_annual_cost(
    baseline_dw_kwh, baseline_water_gal, DISHWASHER_CYCLES_PER_YEAR, include_labor_savings=False
)  # Energy + water for old unit

for model in categories['Category 1: Dishwashers']['Models']:
    annual_cost = calculate_dishwasher_annual_cost(
        model['annual_kWh'],
        model['water_gal_per_cycle'],
        DISHWASHER_CYCLES_PER_YEAR,
        include_labor_savings=False
    )
    annual_savings = baseline_dw_cost - annual_cost
    
    systems_list.append({
        'category': 'Dishwasher',
        'model': model['Model'],
        'lifespan_years': model['expected_lifespan_years'],
        'installed_cost': model['installed_cost_usd'],
        'baseline_annual_cost': baseline_dw_cost,
        'annual_operating_cost': annual_cost,
        'annual_savings': annual_savings,
        'annual_kwh': model['annual_kWh'],
        'water_gal_per_cycle': model['water_gal_per_cycle']
    })

# Process Water Heaters
water_heater_baseline = BASELINES['Water Heater']
baseline_wh_cost = calculate_water_heater_annual_cost(water_heater_baseline['annual_kWh_point'])

for model in categories['Category 2: Water Heaters']['Models']:
    annual_cost = calculate_water_heater_annual_cost(model['annual_kWh'])
    annual_savings = baseline_wh_cost - annual_cost
    
    systems_list.append({
        'category': 'Water Heater',
        'model': model['Model'],
        'lifespan_years': model['expected_lifespan_years'],
        'installed_cost': model['installed_cost_usd'],
        'baseline_annual_cost': baseline_wh_cost,
        'annual_operating_cost': annual_cost,
        'annual_savings': annual_savings,
        'annual_kwh': model['annual_kWh']
    })

# Process Air Conditioners
ac_baseline = BASELINES['Air Conditioner']
baseline_ac_cost = calculate_ac_annual_cost(ac_baseline['annual_kWh_point'])

for model in categories['Category 3: Air Conditioning - Atlanta']['Models']:
    annual_cost = calculate_ac_annual_cost(model['annual_kWh'])
    annual_savings = baseline_ac_cost - annual_cost
    
    systems_list.append({
        'category': 'Air Conditioner',
        'model': model['Model'],
        'lifespan_years': model['expected_lifespan_years'],
        'installed_cost': model['installed_cost_usd'],
        'baseline_annual_cost': baseline_ac_cost,
        'annual_operating_cost': annual_cost,
        'annual_savings': annual_savings,
        'annual_kwh': model['annual_kWh']
    })

# Process LED Lighting
lighting_baseline = BASELINES['Lighting']
baseline_lighting_cost = calculate_ac_annual_cost(lighting_baseline['annual_kWh_point'])

for model in categories['Category 4: Lighting - LED Retrofits']['Models']:
    annual_cost = calculate_ac_annual_cost(model['annual_kWh'])
    annual_savings = baseline_lighting_cost - annual_cost
    
    systems_list.append({
        'category': 'Lighting',
        'model': model['Model'],
        'lifespan_years': model['expected_lifespan_years'],
        'installed_cost': model['installed_cost_usd'],
        'baseline_annual_cost': baseline_lighting_cost,
        'annual_operating_cost': annual_cost,
        'annual_savings': annual_savings,
        'annual_kwh': model['annual_kWh']
    })

# Process Attic Insulation
insulation_baseline = BASELINES['Attic Insulation']
baseline_insulation_cost = calculate_ac_annual_cost(insulation_baseline['annual_kWh_point'])

for model in categories['Category 5: Attic Insulation Upgrades']['Models']:
    # CRITICAL: Model provides annual_kWh_savings (NOT annual_kWh)
    annual_savings = model['annual_kWh_savings'] * ELECTRICITY_RATE
    annual_kwh_post_upgrade = insulation_baseline['annual_kWh_point'] - model['annual_kWh_savings']
    annual_cost = calculate_ac_annual_cost(annual_kwh_post_upgrade)
    
    systems_list.append({
        'category': 'Attic Insulation',
        'model': model['Model'],
        'lifespan_years': model['expected_lifespan_years'],
        'installed_cost': model['installed_cost_usd'],
        'baseline_annual_cost': baseline_insulation_cost,
        'annual_operating_cost': annual_cost,
        'annual_savings': annual_savings,
        'annual_kwh': annual_kwh_post_upgrade,
        'annual_kwh_savings_raw': model['annual_kWh_savings']
    })

# Create DataFrame
df = pd.DataFrame(systems_list)

print(f"Loaded {len(df)} systems")
print(f"  Categories: {df['category'].unique().tolist()}")
print(f"  DataFrame shape: {df.shape}")

# Validate DataFrame
assert len(df) == 11, f"Expected 11 systems, got {len(df)}"
assert df['category'].nunique() == 5, f"Expected 5 categories, got {df['category'].nunique()}"
assert (df['annual_savings'] > 0).all(), "All annual savings should be positive"

# Run reference data validation
if VALIDATION_AVAILABLE:
    print("\nRunning reference data validation...")
    data_result = validate_systems_data()
    if data_result.passed:
        print("  Reference data validation passed")
    else:
        print(f"  Reference data validation failed: {len(data_result.errors)} error(s)")
        for err in data_result.errors[:3]:  # Show first 3 errors
            print(f"    - {err}")
        if len(data_result.errors) > 3:
            print(f"    ... and {len(data_result.errors) - 3} more error(s)")
    if data_result.warnings:
        print(f"  {len(data_result.warnings)} warning(s) - review recommended")

df.head()



Loaded 11 systems
  Categories: ['Dishwasher', 'Water Heater', 'Air Conditioner', 'Lighting', 'Attic Insulation']
  DataFrame shape: (11, 10)

Running reference data validation...
  Reference data validation passed


,category,model,lifespan_years,installed_cost,baseline_annual_cost,annual_operating_cost,annual_savings,annual_kwh,water_gal_per_cycle,annual_kwh_savings_raw
0,Dishwasher,Whirlpool WDF332PAM,10,950,72.88565,46.800848,26.084802,240,3.2,NaN
1,Dishwasher,Bosch 500 Series SHP65CM5N,11,1500,72.88565,46.800848,26.084802,240,3.2,NaN
2,Dishwasher,Miele G 7266 SCVi SF,20,3000,72.88565,42.898348,29.987302,215,3.2,NaN
3,Water Heater,AO Smith Signature 100 Electric Water Heater (...,10,1100,702.45000,679.815500,22.634500,4355,NaN,NaN
4,Water Heater,Rheem ProTerra 50-gal Hybrid Heat Pump,12,2500,702.45000,179.515000,522.935000,1150,NaN,NaN


## 2. Calculate Core Metrics

Calculate payback periods, R/P ratios, and lifetime values for Scenario A (emergency replacement).



In [ ]:
# Calculate payback periods
df['payback_period'] = df.apply(
    lambda row: calculate_payback_period(row['installed_cost'], row['annual_savings']),
    axis=1,
)

finite_paybacks = df.loc[df['payback_period'] != np.inf, 'payback_period']
payback_summary = (
    finite_paybacks.agg(['min', 'max', 'mean'])
    if not finite_paybacks.empty
    else pd.Series({'min': np.nan, 'max': np.nan, 'mean': np.nan})
)

print(f"✓ Payback periods calculated for {len(df)} systems")
if finite_paybacks.empty:
    print("  Range: n/a (all paybacks are infinite)")
    print("  Mean: n/a (all paybacks are infinite)")
else:
    print(f"  Range: {payback_summary['min']:.1f} to {payback_summary['max']:.1f} years")
    print(f"  Mean: {payback_summary['mean']:.1f} years")

# Validate payback periods
payback_valid = (df['payback_period'] > 0) | (df['payback_period'] == np.inf)
assert payback_valid.all(), "Payback periods must be positive or infinite"

positive_savings_mask = df['annual_savings'] > 0
infinite_positive_paybacks = df.loc[positive_savings_mask, 'payback_period'] == np.inf
assert not infinite_positive_paybacks.any(), "Systems with positive savings should not have infinite payback"



✓ Payback periods calculated for 11 systems
  Range: 1.5 to 218.4 years
  Mean: 57.9 years


In [ ]:
# Calculate R/P ratios + sensitivity analysis
df['rp_ratio_scenario_a'] = df.apply(
    lambda row: calculate_rp_ratio(row['lifespan_years'], row['payback_period']),
    axis=1,
)

lifespan_years = df['lifespan_years']
payback_periods = df['payback_period']

# Sensitivity analysis (±20% lifespan variance)
df['rp_ratio_low'] = (lifespan_years * 0.8) / payback_periods
df['rp_ratio_high'] = (lifespan_years * 1.2) / payback_periods

rp_stats = df['rp_ratio_scenario_a'].agg(['min', 'max', 'mean'])
structural_losses = (df['rp_ratio_scenario_a'] < 0.8).sum()
surplus_generators = (df['rp_ratio_scenario_a'] > 1.0).sum()

print(f"✓ R/P ratios calculated for {len(df)} systems")
print(f"  Range: {rp_stats['min']:.3f} to {rp_stats['max']:.3f}")
print(f"  Mean: {rp_stats['mean']:.3f}")
print(f"  Systems with R/P < 0.8 (structural loss): {structural_losses}")
print(f"  Systems with R/P > 1.0 (surplus generator): {surplus_generators}")

# Validate R/P ratios
rp_non_negative = df['rp_ratio_scenario_a'] >= 0
assert rp_non_negative.all(), "R/P ratios must be non-negative"



✓ R/P ratios calculated for 11 systems
  Range: 0.060 to 8.929
  Mean: 1.295
  Systems with R/P < 0.8 (structural loss): 9
  Systems with R/P > 1.0 (surplus generator): 2


In [ ]:
# Calculate cash lifetime values (30-year horizon, Scenario A)
def _compute_cash_metrics(row):
    cash_flow, cash_npv = calculate_lifetime_value_cash(
        row['installed_cost'],
        row['annual_savings'],
        row['lifespan_years'],
        horizon=TIME_HORIZON,
        degradation_rate=EFFICIENCY_DEGRADATION_RATE,
        discount_rate=0.04,
    )
    return pd.Series({
        'lifetime_value_cash_scenario_a': cash_flow[-1],
        'npv_cash_4pct': cash_npv,
    })

cash_metrics = df.apply(_compute_cash_metrics, axis=1)
df[['lifetime_value_cash_scenario_a', 'npv_cash_4pct']] = cash_metrics

cash_values = df['lifetime_value_cash_scenario_a']
cash_npv_values = df['npv_cash_4pct']
negative_cash_mask = cash_values < 0

cash_summary = {
    'mean': cash_values.mean(),
    'npv_mean': cash_npv_values.mean(),
    'negative_count': negative_cash_mask.sum(),
}

# Calculate HELOC lifetime values (30-year horizon, Scenario A)
def _compute_heloc_metrics(row):
    heloc_flow, heloc_npv = calculate_lifetime_value_heloc(
        row['installed_cost'],
        row['annual_savings'],
        row['lifespan_years'],
        rate=HELOC_RATE,
        loan_term=HELOC_LOAN_TERM,
        horizon=TIME_HORIZON,
        degradation_rate=EFFICIENCY_DEGRADATION_RATE,
        discount_rate=0.04,
    )
    return pd.Series({
        'lifetime_value_heloc_scenario_a': heloc_flow[-1],
        'npv_heloc_4pct': heloc_npv,
    })

heloc_metrics = df.apply(_compute_heloc_metrics, axis=1)
df[['lifetime_value_heloc_scenario_a', 'npv_heloc_4pct']] = heloc_metrics

heloc_values = df['lifetime_value_heloc_scenario_a']
heloc_npv_values = df['npv_heloc_4pct']

# Calculate financing amplification factor
cash_zero_mask = np.isclose(cash_values, 0, atol=1e-9)
financing_penalty = heloc_values - cash_values
amplification_factor = np.where(
    negative_cash_mask,
    np.abs(financing_penalty / cash_values),
    np.nan,
)
npv_penalty_pct = np.where(
    cash_zero_mask,
    np.nan,
    (np.abs(cash_values) - np.abs(cash_npv_values)) / np.abs(cash_values) * 100,
)

df['financing_penalty_usd'] = financing_penalty
df['amplification_factor'] = amplification_factor
df['npv_penalty_pct'] = npv_penalty_pct

heloc_summary = {
    'mean': heloc_values.mean(),
    'npv_mean': heloc_npv_values.mean(),
}

print(f"✓ Lifetime values calculated for {len(df)} systems")
print(f"  Cash mean: ${cash_summary['mean']:,.0f}")
print(f"  Cash NPV mean (4%): ${cash_summary['npv_mean']:,.0f}")
print(f"  HELOC mean: ${heloc_summary['mean']:,.0f}")
print(f"  HELOC NPV mean (4%): ${heloc_summary['npv_mean']:,.0f}")
print(f"  Systems with negative cash value: {cash_summary['negative_count']}")



✓ Lifetime values calculated for 11 systems
  Cash mean: $-7,689
  Cash NPV mean (4%): $-5,157
  HELOC mean: $-10,607
  HELOC NPV mean (4%): $-6,078
  Systems with negative cash value: 9


## 3. Comfort Gap Analysis

Calculate comfort gap for environmental utilities (water heaters, AC, insulation).



In [ ]:
# Calculate comfort/utility gap for environmental utilities
df['breakeven_annual_savings'] = df['installed_cost'] / df['lifespan_years']
df['comfort_gap'] = df['breakeven_annual_savings'] - df['annual_savings']
df['comfort_gap_per_hour'] = df['comfort_gap'] / 8760  # Normalize to per-hour cost

# Filter to environmental utilities only
comfort_categories = ['Water Heater', 'Air Conditioner', 'Attic Insulation']
comfort_mask = df['category'].isin(comfort_categories)
environmental_utilities = df[comfort_mask].copy()

if len(environmental_utilities) > 0:
    print("COMFORT GAP ANALYSIS")
    print("=" * 80)
    print(f"Environmental utilities analyzed: {len(environmental_utilities)}")
    print(f"Comfort gap range: ${environmental_utilities['comfort_gap'].min():,.2f} to ${environmental_utilities['comfort_gap'].max():,.2f}/year")
    print(f"Average comfort gap: ${environmental_utilities['comfort_gap'].mean():,.2f}/year")
    print(f"Average R/P ratio: {environmental_utilities['rp_ratio_scenario_a'].mean():.3f}")



COMFORT GAP ANALYSIS
Environmental utilities analyzed: 7
Comfort gap range: $-314.60 to $705.36/year
Average comfort gap: $203.15/year
Average R/P ratio: 0.665


## 4. Validation

Run validation checks on calculated metrics.



In [ ]:
# Validate scenarios for each system
print("=" * 80)
print("VALIDATION: Scenario Results")
print("=" * 80)

for idx, row in df.iterrows():
    scenario_result = {
        'payback_years': row['payback_period'],
        'lifespan': row['lifespan_years'],
        'annual_savings': row['annual_savings'],
        'upfront_cost': row['installed_cost'],
        'net_value': row['lifetime_value_cash_scenario_a'],
        'total_savings': row['lifetime_value_cash_scenario_a'] + row['installed_cost']  # Approximate
    }
    try:
        validate_scenario(scenario_result, f"{row['category']} - {row['model']}")
    except AssertionError as e:
        print(f"⚠ Validation warning: {e}")

print(f"\n✓ Validated {len(df)} systems")
print(f"  All payback periods valid")
print(f"  All R/P ratios non-negative")
print(f"  All annual savings positive")

# Optional: Compare against reference outputs if available
try:
    current_outputs = {
        'rp_ratio_mean': float(df['rp_ratio_scenario_a'].mean()),
        'mean_cash_lifetime_value': float(df['lifetime_value_cash_scenario_a'].mean()),
        'mean_heloc_lifetime_value': float(df['lifetime_value_heloc_scenario_a'].mean()),
        'structural_loss_count': int((df['rp_ratio_scenario_a'] < 0.8).sum()),
    }
    load_reference_and_compare(current_outputs)
    print("✓ Outputs match reference (regression test passed)")
except FileNotFoundError:
    print("ℹ Reference outputs file not found - skipping regression test")
except AssertionError as e:
    print(f"⚠ Regression test failed: {e}")

print("\n" + "=" * 80)
print("✓ Core analysis complete")
print("=" * 80)
print(f"\nDataFrame ready with {len(df)} systems and all calculated metrics.")
print("Use this DataFrame for visualization and Monte Carlo analysis.")



VALIDATION: Scenario Results
⚠ Validation warning: Dishwasher - Whirlpool WDF332PAM: Invalid total_savings (negative): -2118.2192719404366
⚠ Validation warning: Dishwasher - Bosch 500 Series SHP65CM5N: Invalid total_savings (negative): -2269.249006045335
⚠ Validation warning: Dishwasher - Miele G 7266 SCVi SF: Invalid total_savings (negative): -2198.073323773394
⚠ Validation warning: Water Heater - AO Smith Signature 100 Electric Water Heater (50 gal): Invalid total_savings (negative): -2665.0137390629147
⚠ Validation warning: Water Heater - Stiebel Eltron Tempra 24 Plus: Invalid total_savings (negative): -1001.9209871911671
⚠ Validation warning: Air Conditioner - Goodman GSX14 3-Ton Split System: Invalid total_savings (negative): -18263.57235156009
⚠ Validation warning: Air Conditioner - Trane XV18 Variable-Speed Air Conditioner: Invalid total_savings (negative): -19309.519642289517
⚠ Validation warning: Air Conditioner - Mitsubishi Electric MUZ/MSZ-FS18 Mini-Split: Invalid total_savi

c:\Users\grant\OneDrive\Documents\substack\notebooks\lib\replacement_trap_utils.py:470: UserWarning: Dishwasher - Bosch 500 Series SHP65CM5N: Payback period (57.50474931724612 years) exceeds 50 years - verify calculation
  warnings.warn(
c:\Users\grant\OneDrive\Documents\substack\notebooks\lib\replacement_trap_utils.py:470: UserWarning: Dishwasher - Miele G 7266 SCVi SF: Payback period (100.04234458972002 years) exceeds 50 years - verify calculation
  warnings.warn(
c:\Users\grant\OneDrive\Documents\substack\notebooks\lib\replacement_trap_utils.py:470: UserWarning: Air Conditioner - Goodman GSX14 3-Ton Split System: Payback period (218.39147399685532 years) exceeds 50 years - verify calculation
  warnings.warn(
c:\Users\grant\OneDrive\Documents\substack\notebooks\lib\replacement_trap_utils.py:470: UserWarning: Air Conditioner - Trane XV18 Variable-Speed Air Conditioner: Payback period (69.19333829603332 years) exceeds 50 years - verify calculation
  warnings.warn(


In [ ]:
# Validation: Trane XV18 cash vs HELOC (undiscounted vs discounted)
def _format_currency(value):
    sign = "-" if value < 0 else ""
    return f"{sign}${abs(value):,.2f}"


def _npv_penalty(npv_value, undiscounted_value):
    if np.isclose(undiscounted_value, 0, atol=1e-9):
        return np.nan
    return (abs(npv_value) - abs(undiscounted_value)) / abs(undiscounted_value) * 100


def _print_validation_block(name, cumulative_value, npv_value, penalty_pct):
    print(f"{name}:")
    print(f"  - 30-year cumulative (undiscounted): {_format_currency(cumulative_value)}")
    print(f"  - 30-year NPV (4% discount): {_format_currency(npv_value)}")
    if np.isnan(penalty_pct):
        print("  - NPV impact: --")
    elif penalty_pct >= 0:
        print(f"  - NPV makes it {penalty_pct:.1f}% worse (larger magnitude loss)")
    else:
        print(f"  - NPV makes it {abs(penalty_pct):.1f}% less severe (discounting shrinks losses)")
    print()


target_model = "Trane XV18 Variable-Speed Air Conditioner"
trane_row = df[df['model'] == target_model].iloc[0]

cash_flow_trane, cash_npv_trane = calculate_lifetime_value_cash(
    trane_row['installed_cost'],
    trane_row['annual_savings'],
    trane_row['lifespan_years'],
    horizon=TIME_HORIZON,
    degradation_rate=EFFICIENCY_DEGRADATION_RATE,
)
heloc_flow_trane, heloc_npv_trane = calculate_lifetime_value_heloc(
    trane_row['installed_cost'],
    trane_row['annual_savings'],
    trane_row['lifespan_years'],
    rate=HELOC_RATE,
    loan_term=HELOC_LOAN_TERM,
    horizon=TIME_HORIZON,
    degradation_rate=EFFICIENCY_DEGRADATION_RATE,
)

cash_cumulative_trane = cash_flow_trane[-1]
heloc_cumulative_trane = heloc_flow_trane[-1]

cash_penalty_pct_trane = _npv_penalty(cash_npv_trane, cash_cumulative_trane)
heloc_penalty_pct_trane = _npv_penalty(heloc_npv_trane, heloc_cumulative_trane)

print("Trane XV18 Variable-Speed Air Conditioner\n")
_print_validation_block("Cash Purchase", cash_cumulative_trane, cash_npv_trane, cash_penalty_pct_trane)
_print_validation_block("HELOC Financing", heloc_cumulative_trane, heloc_npv_trane, heloc_penalty_pct_trane)

if cash_penalty_pct_trane < 0:
    print("Note: At 4% discounting, the far-future replacement hits (years 15 and 30) shrink faster than the small annual savings,"
          " so the discounted view looks less severe than the raw cumulative loss.\n")
elif cash_penalty_pct_trane > 0:
    print("Note: Discounting magnifies losses because the near-term cash outlay dominates the small, discounted savings stream.\n")



Trane XV18 Variable-Speed Air Conditioner

Cash Purchase:
  - 30-year cumulative (undiscounted): -$31,309.52
  - 30-year NPV (4% discount): -$19,629.62
  - NPV makes it 37.3% less severe (discounting shrinks losses)

HELOC Financing:
  - 30-year cumulative (undiscounted): -$39,709.52
  - 30-year NPV (4% discount): -$22,741.67
  - NPV makes it 42.7% less severe (discounting shrinks losses)

Note: At 4% discounting, the far-future replacement hits (years 15 and 30) shrink faster than the small annual savings, so the discounted view looks less severe than the raw cumulative loss.



In [ ]:
from pathlib import Path

if 'get_notebooks_root' not in globals():
    def get_notebooks_root():
        """Fallback path resolver so this cell can run without the setup cell."""
        cwd = Path.cwd()

        if (cwd.parent / 'lib' / 'replacement_trap_utils.py').exists() and cwd.name == 'analysis':
            return cwd.parent
        if (cwd / 'lib' / 'replacement_trap_utils.py').exists():
            return cwd
        if (cwd / 'notebooks' / 'lib' / 'replacement_trap_utils.py').exists():
            return cwd / 'notebooks'
        for parent in [cwd] + list(cwd.parents):
            if (parent / 'lib' / 'replacement_trap_utils.py').exists():
                return parent

        return cwd.parent if cwd.name == 'analysis' else cwd

# Save DataFrame for use in other notebooks
notebooks_root = get_notebooks_root()
df_save_path = notebooks_root / 'data' / 'replacement_trap_df.pkl'
df.to_pickle(df_save_path)
print(f"DataFrame saved to {df_save_path}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {len(df.columns)}")


DataFrame saved to c:\Users\grant\OneDrive\Documents\substack\notebooks\data\replacement_trap_df.pkl
  Shape: (11, 24)
  Columns: 24
